# Phase 2: generate the guard's training data, on a free Colab T4

Turns the 720 English seed payloads (`corpus/guardset/seed/`) into Hindi, Romanized Hindi and
code-mixed text, and writes the benign side (ordinary carrier texts and adversarial-benign texts) in
all four languages. docs/v2-plan.md, Phase 2 steps 5-6.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all.** Every model call is cached in
`corpus/guardset/drafts/work/` and saved to Drive every 10 minutes and after each stage, so a
disconnected or out-of-quota session resumes where it stopped (losing at most 10 minutes): re-run all
cells and nothing already done is sent again.

In order:
1. GPU check, config, Drive, repo (public, branch `v2`), install
2. **IndicTrans2** English→Hindi drafts of the payloads (optional; falls back to the LLM alone)
3. the **Qwen** server (vLLM), then the generator: payloads → hi → hi_latn, mixed; carrier texts; adversarial-benign texts
4. the report, and `guardset_drafts.tgz` to download

Bring `guardset_drafts.tgz` back to the laptop; the dataset is built and audited there
(`python -m corpus.guardset.build`). Nothing here is paid: open-weight models on the free GPU.

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"

## 2 · Config

The only cell you should need to edit.

In [ ]:
MODEL      = "Qwen/Qwen2.5-14B-Instruct-AWQ"   # Apache-2.0; the v1 agent model, known to load on a T4
QUANT      = "awq"
FALLBACK_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MAX_LEN    = 4096          # prompts are short; a small context leaves room for many parallel requests
GPU_UTIL   = 0.92
VLLM_SPEC  = "vllm==0.6.6.post1"               # pinned for Turing, as in the v1 notebooks

USE_INDICTRANS2 = True     # English->Hindi drafts before the LLM rewrite (the plan's route)
INDICTRANS2     = "ai4bharat/indictrans2-en-indic-dist-200M"   # MIT; the 1B model also works if memory allows
# IndicTrans2 is gated: accept its terms once at https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M
# (logged in), then add your Hugging Face token as a Colab secret named HF_TOKEN (key icon, left bar),
# with "Notebook access" switched on. Read access is enough.

REPO_DIR  = "/content/agentfw"
GIT_URL   = "https://github.com/Sameerkc3579/bhasha-shield.git"
BRANCH    = "v2"
DRIVE_DIR = "/content/drive/MyDrive/agentfw_guardset"
USE_DRIVE = True
WORK      = "corpus/guardset/drafts/work"
print("model:", MODEL, "| IndicTrans2:", INDICTRANS2 if USE_INDICTRANS2 else "off")

## 3 · Drive

Keeps the generation cache between sessions.

In [ ]:
import os
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        DRIVE_OK = True
        print("drive ready:", DRIVE_DIR, os.listdir(DRIVE_DIR) or "(empty)")
    except Exception as e:
        print("drive unavailable, continuing without persistence:", e)

## 4 · Repo and cache

In [ ]:
import os, subprocess, threading
# Always the newest code of the branch, also when this runtime still has an older copy from an earlier run.
if os.path.exists(f"{REPO_DIR}/pyproject.toml"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, GIT_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
assert "KEEP_PASSED" in open("corpus/guardset/generate.py", encoding="utf-8").read(), \
    "old code on GitHub: the generator fixes have not been pushed yet"

TGZ = f"{DRIVE_DIR}/guardset_drafts.tgz"
if os.path.exists(WORK):
    print("cache already here from this runtime: not restoring the (older) Drive copy over it")
elif DRIVE_OK and os.path.exists(TGZ):
    subprocess.run(["tar", "xzf", TGZ, "-C", REPO_DIR], check=True)
    print("restored the generation cache from", TGZ)
else:
    print("no cache in Drive: starting cold (expected the first time)")

_save_lock = threading.Lock()
def save(note=""):
    # copy then rename, so a session killed mid-copy never leaves a half-written archive
    with _save_lock:
        subprocess.run(["tar", "czf", "/content/guardset_drafts.tgz", "corpus/guardset/drafts"], check=False)
        size = os.path.getsize("/content/guardset_drafts.tgz") / 1e6
        if DRIVE_OK:
            subprocess.run(["cp", "/content/guardset_drafts.tgz", TGZ + ".tmp"], check=True)
            os.replace(TGZ + ".tmp", TGZ)
    print(f"saved{' (' + note + ')' if note else ''}: guardset_drafts.tgz {size:.1f} MB", "-> Drive" if DRIVE_OK else "")

## 5 · Install

A separate Python 3.10 environment (Colab's own Python is too new for the vLLM pinned for the T4).
First time 5-10 minutes; if it suggests restarting the runtime, **do not**: just continue.

In [ ]:
import subprocess, sys, os
VENV = "/content/venv"
PY = f"{VENV}/bin/python"

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print("".join((r.stdout + r.stderr).splitlines(keepends=True)[-40:]))
        raise RuntimeError(f"failed: {' '.join(cmd)}")
    return r.stdout

def stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"})
    for line in p.stdout:
        print(line, end="")
    return p.wait()

if not os.path.exists(PY):
    run(["python3", "-m", "pip", "install", "-q", "uv"])
    run(["python3", "-m", "uv", "venv", "--seed", "--python", "3.10", VENV])
pip = lambda *a: run([PY, "-m", "pip", "install", "-q", *a])
pip(VLLM_SPEC, "transformers>=4.45.2,<4.48")      # transformers 5.x breaks this vLLM's tokenizer
pip("--no-deps", "-e", ".")
pip("pydantic>=2.7", "requests", "sentencepiece", "sacremoses", "huggingface_hub")
print(run([PY, "-c", "import vllm, transformers; print('vllm', vllm.__version__, '| transformers', transformers.__version__)"]))

## 6 · English seeds

The 720 payloads, rebuilt from the repo (the same on every machine).

In [ ]:
assert stream([PY, "-m", "corpus.guardset.seed", "export"]) == 0

## 7 · IndicTrans2 drafts (optional)

English→Hindi drafts that the LLM then rewrites for natural style. If this cell fails, set
`USE_INDICTRANS2 = False` in cell 2 and re-run: the LLM then translates directly, and the report says so.
It runs **before** the Qwen server because both do not fit on one T4 at once.

In [ ]:
import json, os
DRAFTS_HI = f"{WORK}/indictrans2_hi.jsonl"
if USE_INDICTRANS2 and not os.path.exists(DRAFTS_HI):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # the subprocess below inherits it
        print("HF_TOKEN: found")
    except Exception as e:
        print("HF_TOKEN: not found (" + type(e).__name__ + "); IndicTrans2 is gated and will fail without it")
    # Constrained to what is installed, so the toolkit cannot pull in a transformers that breaks vLLM.
    open("/content/pins.txt", "w").write(run([PY, "-m", "pip", "freeze", "--exclude-editable"]))
    try:
        pip("-c", "/content/pins.txt", "IndicTransToolkit")
    except RuntimeError as e:
        print("IndicTransToolkit did not install:", e)
    script = r'''
import json, sys, re, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor
name, out = sys.argv[1], sys.argv[2]
tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(name, trust_remote_code=True, torch_dtype=torch.float16).cuda().eval()
ip = IndicProcessor(inference=True)
rows = [json.loads(l) for l in open("corpus/guardset/drafts/seed_en.jsonl", encoding="utf-8")]
res = []
for i in range(0, len(rows), 16):
    chunk = rows[i:i + 16]
    # line by line, so the line structure of the template survives
    lines = [(j, ln) for j, r in enumerate(chunk) for ln in r["text"].split("\n")]
    src = [ln for _, ln in lines if ln.strip()]
    batch = ip.preprocess_batch(src, src_lang="eng_Latn", tgt_lang="hin_Deva")
    enc = tok(batch, padding="longest", truncation=True, max_length=256, return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = model.generate(**enc, max_length=256, num_beams=4)
    dec = ip.postprocess_batch(tok.batch_decode(gen, skip_special_tokens=True), lang="hin_Deva")
    it = iter(dec)
    per = [[] for _ in chunk]
    for j, ln in lines:
        per[j].append(next(it) if ln.strip() else "")
    for r, p in zip(chunk, per):
        res.append({"seed_id": r["seed_id"], "text": "\n".join(p), "model": name})
with open(out, "w", encoding="utf-8") as f:
    for r in res:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(res), "drafts ->", out)
'''
    os.makedirs(WORK, exist_ok=True)
    open("/content/it2.py", "w").write(script)
    if stream([PY, "/content/it2.py", INDICTRANS2, DRAFTS_HI]) != 0:
        print("IndicTrans2 failed. If the error says 'gated repo': accept the terms on the model page and add the")
        print("HF_TOKEN secret (see cell 2), then re-run this cell. Otherwise the LLM translates alone;")
        print("set USE_INDICTRANS2 = False to skip this cell.")
    else:
        save("IndicTrans2 drafts")
print("drafts:", DRAFTS_HI if os.path.exists(DRAFTS_HI) else "none (LLM-only route)")

## 8 · Serve Qwen (vLLM)

First boot downloads ~10 GB. The log is `/content/vllm.log`.

In [ ]:
import subprocess, time, requests, os
LOG = "/content/vllm.log"
def serve(model, quant):
    cmd = [PY, "-m", "vllm.entrypoints.openai.api_server", "--model", model, "--dtype", "half",
           "--max-model-len", str(MAX_LEN), "--gpu-memory-utilization", str(GPU_UTIL),
           "--disable-log-requests", "--port", "8000", "--enforce-eager"] + (["--quantization", quant] if quant else [])
    return subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
def up(proc, minutes=25):
    end = time.time() + minutes * 60
    while time.time() < end and proc.poll() is None:
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=3).ok:
                return True
        except Exception:
            pass
        time.sleep(10)
    return False
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"]); time.sleep(5)
SERVED = MODEL
server = serve(MODEL, QUANT)
if not up(server):
    print("".join(open(LOG).readlines()[-30:]))
    server.kill(); SERVED = FALLBACK_MODEL
    print("falling back to", SERVED, "(record this in RESULTS.md)")
    server = serve(FALLBACK_MODEL, "awq")
    assert up(server), "no server: read /content/vllm.log"
from huggingface_hub import model_info
REV = model_info(SERVED).sha
print("SERVED =", SERVED, "| revision", REV)
open(f"{WORK}/model.json", "w").write(json.dumps({"model": SERVED, "revision": REV,
                                                   "indictrans2": INDICTRANS2 if os.path.exists(DRAFTS_HI) else None}))

## 9 · Generate

Payloads (hi, then hi_latn and mixed), carrier texts, adversarial-benign texts, then assembly. Every
output is checked (amounts, ids and links must survive; right script; no chatter) and retried twice;
what still fails is counted in the report, never used. Saved to Drive every 10 minutes and after each
stage. About 1 hour from cold; a rerun sends only what is missing.

`--keep-passed`: when a prompt has changed (the firmer code-mixed instruction of 27 Nov), rows that
already passed their checks are kept and only the rejected ones are sent again.

In [ ]:
args = ["--base-url", "http://localhost:8000/v1", "--model", SERVED, "--work", WORK, "--keep-passed"]
if os.path.exists(DRAFTS_HI):
    args += ["--drafts-hi", DRAFTS_HI]
stop = threading.Event()
def every_10_minutes():
    while not stop.wait(600):
        save("periodic")
threading.Thread(target=every_10_minutes, daemon=True).start()
try:
    for stage in ("payloads", "bodies", "advbenign", "assemble"):
        print(f"\n=== {stage} ===")
        rc = stream([PY, "-m", "corpus.guardset.generate", stage, *args])
        save(stage)
        assert rc == 0, f"{stage} failed: re-run this cell to resume"
finally:
    stop.set()

## 10 · Report and download

In [ ]:
import json
rep = json.load(open(f"{WORK}/generate_report.json"))
print(json.dumps(rep, indent=1))
save("final")
try:
    from google.colab import files
    files.download("/content/guardset_drafts.tgz")
except Exception:
    print("download /content/guardset_drafts.tgz from the file browser (or from Drive:", TGZ, ")")